<a href="https://colab.research.google.com/github/Lthao-stack/Al-ve-suc-khoe-gioi-/blob/main/chuyende2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio scikit-learn pandas -q

import gradio as gr
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

APP_NAME = "TRỢ LÝ ẢO HỖ TRỢ TRẢ LỜI CÁC CÂU HỎI THƯỜNG GẶP VỀ SỨC KHỎE GIỚI TÍNH"

# =========================
# 1. CƠ SỞ TRI THỨC MỞ RỘNG
# =========================

knowledge_base = [
    {"topic": "dậy thì bình thường tuổi dậy thì nam nữ", "answer": "Dậy thì là giai đoạn cơ thể phát triển từ trẻ em sang người trưởng thành. Ở nữ thường bắt đầu khoảng 8 đến 13 tuổi, ở nam thường khoảng 9 đến 14 tuổi. Nữ có thể phát triển ngực, mọc lông, thay đổi vóc dáng và xuất hiện kinh nguyệt. Nam có thể vỡ giọng, cao nhanh, mọc lông, cơ bắp phát triển, cơ quan sinh dục thay đổi và có thể có mộng tinh. Mỗi người phát triển với tốc độ khác nhau nên không nên quá lo nếu chỉ chênh lệch nhẹ."},

    {"topic": "thủ dâm có hại không", "answer": "Thủ dâm là hiện tượng sinh lý có thể gặp ở tuổi dậy thì và không phải lúc nào cũng có hại. Tuy nhiên, nếu lạm dụng đến mức ảnh hưởng học tập, giấc ngủ, sức khỏe, tâm lý hoặc sinh hoạt hằng ngày thì nên điều chỉnh lại. Nếu cảm thấy lo lắng, ám ảnh hoặc khó kiểm soát hành vi, nên chia sẻ với người lớn đáng tin cậy hoặc chuyên gia tư vấn."},

    {"topic": "có thai khi cọ xát bên ngoài mặc quần áo", "answer": "Khả năng có thai khi chỉ cọ xát bên ngoài hoặc vẫn mặc nguyên quần áo thường rất thấp. Tuy nhiên, nguy cơ có thể tăng nếu tinh dịch tiếp xúc trực tiếp gần hoặc vào âm đạo. Nếu lo lắng sau một tình huống có nguy cơ, nên theo dõi chu kỳ kinh và dùng que thử thai sau thời điểm phù hợp."},

    {"topic": "quan hệ tình dục an toàn bao cao su", "answer": "Quan hệ tình dục an toàn là việc bảo vệ sức khỏe, tôn trọng sự đồng thuận và giảm nguy cơ mang thai ngoài ý muốn cũng như bệnh lây truyền qua đường tình dục. Bao cao su là biện pháp thường được khuyến nghị vì vừa hỗ trợ tránh thai vừa giúp giảm nguy cơ bệnh lây truyền qua đường tình dục."},

    {"topic": "thuốc tránh thai khẩn cấp", "answer": "Thuốc tránh thai khẩn cấp là biện pháp dùng sau tình huống có nguy cơ mang thai ngoài ý muốn, nhưng chỉ nên xem là giải pháp tạm thời. Không nên lạm dụng vì có thể gây rối loạn kinh nguyệt, buồn nôn, mệt mỏi hoặc ảnh hưởng nội tiết. Nếu cần dùng, nên đọc kỹ hướng dẫn thuốc hoặc hỏi dược sĩ, nhân viên y tế."},

    {"topic": "quên uống thuốc tránh thai hằng ngày", "answer": "Khi quên uống thuốc tránh thai hằng ngày, nên uống bổ sung ngay khi nhớ ra và đọc kỹ hướng dẫn trên vỉ thuốc vì cách xử lý có thể khác nhau tùy loại thuốc và số viên đã quên. Nếu không chắc chắn, nên hỏi dược sĩ hoặc nhân viên y tế để tránh giảm hiệu quả tránh thai."},

    {"topic": "dấu hiệu bệnh lây truyền qua đường tình dục sti", "answer": "Một số dấu hiệu có thể gợi ý bệnh lây truyền qua đường tình dục gồm ngứa rát vùng kín, đau khi đi tiểu, nổi mụn hoặc vết loét bất thường, dịch tiết có mùi lạ, đau vùng bụng dưới hoặc chảy máu bất thường. Tuy nhiên, một số bệnh có thể không có triệu chứng rõ ràng, vì vậy nếu có nguy cơ thì nên đi khám để được xét nghiệm và tư vấn."},

    {"topic": "mang thai ngoài ý muốn dấu hiệu có thai que thử thai", "answer": "Dấu hiệu có thai có thể gồm chậm kinh, căng tức ngực, buồn nôn, mệt mỏi hoặc thay đổi cảm giác ăn uống. Tuy nhiên, các dấu hiệu này không đủ để khẳng định chắc chắn. Có thể dùng que thử thai sau khi quan hệ khoảng 7 đến 14 ngày hoặc sau khi trễ kinh để có kết quả đáng tin cậy hơn."},

    {"topic": "quan hệ lần đầu đau chảy máu", "answer": "Quan hệ lần đầu có thể đau hoặc có ít máu ở một số người, nhưng không phải ai cũng giống nhau. Mức độ đau hoặc chảy máu phụ thuộc vào tâm lý, sự thoải mái, cấu tạo cơ thể và mức độ sẵn sàng. Nếu đau nhiều, chảy máu nhiều hoặc kéo dài thì nên đi khám."},

    {"topic": "hòa hợp đời sống vợ chồng chăn gối", "answer": "Sự hòa hợp trong đời sống vợ chồng cần sự chia sẻ thẳng thắn, tôn trọng cảm xúc, lắng nghe nhu cầu của nhau và không gây áp lực. Nếu có khó khăn kéo dài, hai vợ chồng có thể tìm đến chuyên gia tư vấn hôn nhân hoặc bác sĩ chuyên khoa."},

    {"topic": "gay lesbian bisexual xu hướng tính dục", "answer": "Xu hướng tính dục là cảm xúc hấp dẫn tình cảm hoặc tình dục của một người với người khác. Một số người có thể là dị tính, đồng tính hoặc song tính. Việc cảm thấy bối rối trong tuổi dậy thì hoặc chưa xác định rõ xu hướng của mình là điều có thể xảy ra và không cần phải quá áp lực."},

    {"topic": "come out lộ diện với gia đình", "answer": "Come-out là việc chia sẻ xu hướng tính dục hoặc bản dạng giới với người khác. Việc này nên được thực hiện khi bạn cảm thấy an toàn, ổn định tâm lý và có người hỗ trợ đáng tin cậy. Không nên tự gây áp lực cho bản thân nếu chưa sẵn sàng."},

    {"topic": "người chuyển giới dùng hormone", "answer": "Người chuyển giới cần được tư vấn y khoa trước khi sử dụng hormone vì thuốc có thể gây tác dụng phụ nếu dùng sai cách. Không nên tự ý mua hormone qua mạng hoặc dùng theo truyền miệng mà không có hướng dẫn chuyên môn."},

    {"topic": "xuất tinh sớm", "answer": "Xuất tinh sớm là tình trạng khá phổ biến ở nam giới và có thể liên quan đến áp lực tâm lý, căng thẳng, lo âu hoặc thói quen sinh hoạt. Nhiều trường hợp có thể cải thiện bằng thay đổi lối sống, giảm stress hoặc hỗ trợ y khoa phù hợp."},

    {"topic": "rối loạn cương dương", "answer": "Rối loạn cương dương có thể liên quan đến stress, thiếu ngủ, áp lực tâm lý, sử dụng chất kích thích hoặc bệnh lý nền. Nếu tình trạng kéo dài và ảnh hưởng cuộc sống, nên đi khám để được hỗ trợ đúng cách."},

    {"topic": "khô rát đau khi quan hệ nữ giới", "answer": "Đau hoặc khô rát khi quan hệ ở nữ có thể liên quan đến thiếu sự chuẩn bị tâm lý, căng thẳng, thay đổi hormone hoặc viêm nhiễm. Nếu tình trạng kéo dài hoặc gây đau nhiều thì nên đi khám phụ khoa."},

    {"topic": "không có khoái cảm suy giảm ham muốn", "answer": "Suy giảm ham muốn hoặc khó đạt khoái cảm có thể liên quan đến stress, mất ngủ, áp lực học tập hoặc công việc, trầm cảm, thay đổi hormone hoặc thiếu giao tiếp trong mối quan hệ. Nếu kéo dài và gây lo lắng, nên tìm hỗ trợ chuyên môn."},

    {"topic": "đồng thuận trong tình dục", "answer": "Đồng thuận là khi cả hai người đều tự nguyện, tỉnh táo, thoải mái và đồng ý rõ ràng. Đồng thuận có thể thay đổi hoặc bị rút lại bất cứ lúc nào. Không ai được ép buộc, gây áp lực, đe dọa hoặc lợi dụng người khác."},

    {"topic": "từ chối bạn tình đặt giới hạn cá nhân", "answer": "Bạn có quyền từ chối những điều khiến bản thân không thoải mái. Việc từ chối nên rõ ràng, bình tĩnh và tôn trọng. Một mối quan hệ lành mạnh cần tôn trọng giới hạn cá nhân của nhau."},

    {"topic": "bị lạm dụng tình dục tổn thương tâm lý", "answer": "Người từng bị lạm dụng hoặc tổn thương tình dục có thể gặp lo âu, sợ hãi hoặc ám ảnh kéo dài. Đây không phải lỗi của nạn nhân. Việc tìm sự hỗ trợ từ chuyên gia tâm lý, người thân đáng tin cậy hoặc cơ sở hỗ trợ là rất quan trọng."},

    {"topic": "dung dịch vệ sinh hằng ngày", "answer": "Dung dịch vệ sinh có thể dùng nếu chọn loại phù hợp và dùng đúng cách. Không nên thụt rửa sâu bên trong âm đạo vì có thể làm mất cân bằng môi trường tự nhiên và tăng nguy cơ viêm nhiễm. Nếu có ngứa, rát, mùi lạ hoặc dịch bất thường thì nên đi khám."},

    {"topic": "kích thước cậu nhỏ bình thường", "answer": "Kích thước cơ quan sinh dục ở mỗi người là khác nhau và không có một tiêu chuẩn hoàn hảo tuyệt đối. Nhiều lo lắng xuất phát từ phim ảnh hoặc mạng xã hội. Điều quan trọng hơn là sức khỏe, sự tự tin và sự tôn trọng trong mối quan hệ."},

    {"topic": "hình dáng cô bé bình thường", "answer": "Hình dáng vùng kín ở nữ có sự đa dạng tự nhiên giữa mỗi người. Không có một hình mẫu duy nhất nào được xem là chuẩn. Nếu không có đau, ngứa, dịch bất thường hoặc tổn thương thì thường không cần quá lo lắng."}
]

df = pd.DataFrame(knowledge_base)
documents = (df["topic"] + " " + df["answer"]).tolist()

vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(documents)

# =========================
# 2. TỪ KHÓA
# =========================

greeting_patterns = [r"^(xin chào|chào|hello|hi|hey|bot ơi)[\s!,.]*$"]

end_keywords = [
    "tạm biệt", "bye", "kết thúc", "dừng", "hết rồi",
    "không hỏi nữa", "xong rồi", "ok cảm ơn", "cảm ơn", "thank"
]

advice_keywords = [
    "nên làm gì", "làm sao", "làm thế nào", "có sao không",
    "nguy hiểm không", "có cần đi khám không", "tư vấn",
    "lời khuyên", "lo quá", "sợ quá"
]

warning_keywords = [
    "đau dữ dội", "chảy máu nhiều", "chảy máu bất thường",
    "ngứa rát", "dịch lạ", "mùi hôi", "mủ", "đau khi đi tiểu",
    "ngất", "bị ép", "ép buộc", "xâm hại", "lạm dụng",
    "không an toàn", "tự ý mua hormone"
]

# =========================
# 3. HÀM XỬ LÝ
# =========================

def is_greeting(text):
    text = text.lower().strip()
    if len(text.split()) > 4:
        return False
    return any(re.match(pattern, text) for pattern in greeting_patterns)

def is_end_conversation(text):
    return any(kw in text.lower() for kw in end_keywords)

def user_need_advice(text):
    return any(kw in text.lower() for kw in advice_keywords)

def classify_topic_agent(text):
    text = text.lower()

    if any(kw in text for kw in ["gay", "lesbian", "bisexual", "lgbt", "come out", "đồng tính", "song tính", "chuyển giới"]):
        return "LGBTQ+ và bản dạng giới"

    if any(kw in text for kw in ["dậy thì", "vỡ tiếng", "vỡ giọng", "mọc lông", "mộng tinh", "ngực", "thủ dâm"]):
        return "Tuổi dậy thì và phát triển cơ thể"

    if any(kw in text for kw in ["bao cao su", "tránh thai", "thuốc tránh thai", "cọ xát", "có thai", "que thử thai"]):
        return "Tình dục an toàn và tránh thai"

    if any(kw in text for kw in ["sti", "bệnh lây", "ngứa", "rát", "dịch", "mùi hôi", "viêm nhiễm"]):
        return "Sức khỏe sinh sản và bệnh lý"

    if any(kw in text for kw in ["xuất tinh sớm", "rối loạn cương", "khô rát", "khoái cảm", "ham muốn"]):
        return "Chức năng tình dục"

    if any(kw in text for kw in ["đồng thuận", "từ chối", "lạm dụng", "tổn thương", "ép buộc"]):
        return "Tâm lý, mối quan hệ và sự đồng thuận"

    if any(kw in text for kw in ["dung dịch vệ sinh", "vệ sinh", "vùng kín", "cậu nhỏ", "cô bé", "kích thước"]):
        return "Vệ sinh và chăm sóc vùng nhạy cảm"

    if any(kw in text for kw in ["lần đầu", "chảy máu", "vợ chồng", "chăn gối", "hòa hợp"]):
        return "Tình dục học và hôn nhân"

    return "Sức khỏe giới tính chung"

def retrieve_answer(user_question):
    user_vec = vectorizer.transform([user_question])
    scores = cosine_similarity(user_vec, tfidf_matrix).flatten()
    best_idx = scores.argmax()
    best_score = scores[best_idx]

    if best_score < 0.08:
        return None

    return df.iloc[best_idx]["answer"]

def generate_advice(text):
    text = text.lower()
    advice = []

    if "mụn" in text:
        advice += ["- Rửa mặt nhẹ nhàng.", "- Không tự ý nặn mụn.", "- Nếu mụn viêm nặng nên khám da liễu."]

    if "kinh" in text or "trễ kinh" in text:
        advice += ["- Theo dõi chu kỳ kinh nguyệt.", "- Nếu trễ kinh kéo dài hoặc đau nhiều nên đi khám."]

    if "tránh thai" in text or "có thai" in text:
        advice += ["- Không tự xử lý theo thông tin không rõ nguồn.", "- Nên hỏi dược sĩ hoặc nhân viên y tế nếu không chắc chắn."]

    if "lgbt" in text or "come out" in text or "đồng tính" in text:
        advice += ["- Không cần vội gắn nhãn bản thân.", "- Chỉ chia sẻ khi cảm thấy an toàn và có người hỗ trợ."]

    if "hormone" in text:
        advice += ["- Không tự ý mua hoặc dùng hormone.", "- Cần được bác sĩ chuyên khoa tư vấn và theo dõi."]

    if "lạm dụng" in text or "xâm hại" in text or "ép buộc" in text:
        advice += ["- Đây không phải lỗi của nạn nhân.", "- Nên tìm người lớn đáng tin cậy hoặc chuyên gia tâm lý để được hỗ trợ."]

    if not advice:
        return ""

    return "\n\nLời khuyên:\n" + "\n".join(advice)

def generate_warning(text):
    text = text.lower()

    if any(kw in text for kw in warning_keywords):
        return (
            "\n\nCảnh báo:\n"
            "- Câu hỏi có dấu hiệu cần được quan tâm thêm.\n"
            "- Nếu tình trạng kéo dài, nặng hơn hoặc gây lo lắng nhiều, nên trao đổi với người lớn đáng tin cậy hoặc đến cơ sở y tế.\n"
            "- Nếu có yếu tố bị ép buộc, xâm hại hoặc không an toàn, cần tìm sự hỗ trợ ngay."
        )

    return ""

def generate_notice(text):
    return (
        "\n\nLưu ý chung:\n"
        "Thông tin trên chỉ mang tính tham khảo, không thay thế cho tư vấn y tế chuyên môn."
    )

# =========================
# 4. BỘ NHỚ HỘI THOẠI
# =========================

asked_questions = []

def is_repeated_question(message):
    if not asked_questions:
        return False

    all_texts = asked_questions + [message]
    temp_vec = vectorizer.transform(all_texts)
    scores = cosine_similarity(temp_vec[-1], temp_vec[:-1]).flatten()

    return scores.max() > 0.88

# =========================
# 5. AGENTIC AI ARCHITECTURE
# =========================

class InputAnalysisAgent:
    def run(self, message):
        text = message.lower().strip()
        return {
            "text": text,
            "topic_group": classify_topic_agent(text),
            "is_greeting": is_greeting(text),
            "is_end": is_end_conversation(text),
            "need_advice": user_need_advice(text)
        }

class ConversationAgent:
    def run(self, text):
        global asked_questions

        if is_end_conversation(text):
            asked_questions = []
            return "end"

        if is_repeated_question(text):
            return "repeat"

        asked_questions.append(text)
        return "continue"

class RetrievalAgent:
    def run(self, text):
        return retrieve_answer(text)

class SafetyAgent:
    def run(self, text):
        return generate_warning(text)

class AdviceAgent:
    def run(self, text, need_advice):
        return generate_advice(text) if need_advice else ""

class ResponseGenerationAgent:
    def run(self, topic_group, answer, advice, warning):
        response = (
            f"Nhóm nội dung: {topic_group}\n\n"
            f"Nhận định:\n{answer}"
        )

        response += advice
        response += warning
        response += generate_notice(answer)

        return response

# =========================
# 6. CHATBOT
# =========================

def chatbot(message, history):
    if not message or not message.strip():
        return "Bạn hãy nhập câu hỏi để mình hỗ trợ nhé."

    input_agent = InputAnalysisAgent()
    conversation_agent = ConversationAgent()
    retrieval_agent = RetrievalAgent()
    safety_agent = SafetyAgent()
    advice_agent = AdviceAgent()
    response_agent = ResponseGenerationAgent()

    analysis = input_agent.run(message)
    text = analysis["text"]

    if analysis["is_greeting"]:
        return (
            "Xin chào 👋\n\n"
            "Mình là trợ lý ảo AI hỗ trợ trả lời các câu hỏi thường gặp về sức khỏe giới tính.\n\n"
            "Bạn có thể đặt câu hỏi, mình sẽ giải thích theo cách dễ hiểu và phù hợp."
        )

    status = conversation_agent.run(text)

    if status == "end":
        return (
            "Cuộc trò chuyện đã được kết thúc.\n\n"
            "Mình đã làm mới ngữ cảnh để cuộc trò chuyện tiếp theo được xử lý độc lập.\n\n"
            "Cảm ơn bạn đã sử dụng trợ lý ảo."
        )

    if status == "repeat":
        return (
            "Bạn đã hỏi nội dung tương tự trước đó rồi.\n\n"
            "Bạn có thể hỏi chi tiết khác hoặc chuyển sang nội dung mới nhé."
        )

    answer = retrieval_agent.run(text)

    if answer is None:
        return (
            "Mình chưa tìm thấy thông tin thật sự phù hợp trong cơ sở tri thức hiện tại.\n\n"
            "Bạn có thể diễn đạt rõ hơn để mình hỗ trợ chính xác hơn."
        )

    advice = advice_agent.run(text, analysis["need_advice"])
    warning = safety_agent.run(text)

    return response_agent.run(
        analysis["topic_group"],
        answer,
        advice,
        warning
    )

# =========================
# 7. GIAO DIỆN GRADIO
# =========================

custom_css = """
.gradio-container {
    background: #111827 !important;
    color: white !important;
}

.chatbot {
    border-radius: 16px !important;
}

textarea {
    border-radius: 12px !important;
}
"""

demo = gr.ChatInterface(
    fn=chatbot,
    title=APP_NAME,
    description="",
    textbox=gr.Textbox(
        placeholder="Nhập câu hỏi của bạn tại đây...",
        container=True,
        scale=7
    ),
    examples=[],
    css=custom_css
)

demo.launch(share=True)